# Palm oil price shocks and deforestation: exposure design

## 1. Data and outcome

In [26]:
import numpy as np
import pandas as pd

BASE = "https://raw.githubusercontent.com/AndrewEberhardt/DataScienceProject/main/data/"
START_YEAR, END_YEAR = 2001, 2023  # DeDuCE coverage

In [27]:
# Load the 62-country GFW panel, DeDuCE commodity attribution and Pink Sheet prices
panel_raw = pd.read_csv(BASE + "processed/shock_panel.csv")
deduce = pd.read_csv(BASE + "processed/deduce_selected_commodities.csv")
prices = pd.read_csv(BASE + "processed/commodity_prices_annual.csv")

# Palm oil price, kept for the next steps (one lag requires START_YEAR - 1)
palm_price = prices.loc[prices["commodity"] == "Palm oil", ["year", "price_real", "log_price_real"]]
palm_price_years = set(palm_price.dropna()["year"])
assert set(range(START_YEAR - 1, END_YEAR + 1)) <= palm_price_years

print("GFW panel:", panel_raw.shape, "| countries:", panel_raw["iso3"].nunique(),
      "| years:", panel_raw["year"].min(), "-", panel_raw["year"].max())
print("Palm oil price years:", palm_price["year"].min(), "-", palm_price["year"].max())

GFW panel: (1550, 13) | countries: 62 | years: 2001 - 2025
Palm oil price years: 1960 - 2025


In [28]:
# Restrict to the DeDuCE period and attach oil palm deforestation on iso3 + year
panel = panel_raw[panel_raw["year"].between(START_YEAR, END_YEAR)].copy()
palm_defor = deduce.loc[deduce["commodity"] == "Oil palm fruit", ["iso3", "year", "defor_ha"]]

df = panel.merge(palm_defor, on=["iso3", "year"], how="left", validate="one_to_one")

# Countries absent from the oil palm rows of DeDuCE are coded as zero and flagged
df["in_deduce_palm"] = df["defor_ha"].notna()
df["defor_ha"] = df["defor_ha"].fillna(0.0)

assert df["iso3"].nunique() == 62
assert len(df) == 62 * (END_YEAR - START_YEAR + 1)
assert df["extent_2000_ha"].gt(0).all()

absent = df.loc[~df["in_deduce_palm"], ["iso3", "country", "region"]].drop_duplicates()
all_deduce_iso3 = set(deduce["iso3"])
assert set(absent["iso3"]) <= all_deduce_iso3  # present in DeDuCE for other commodities
print("Countries with no oil palm rows in DeDuCE:", len(absent))
print(absent.sort_values(["region", "country"]).to_string(index=False))

Countries with no oil palm rows in DeDuCE: 18
iso3      country                    region
 ETH     Ethiopia                    Africa
 KEN        Kenya                    Africa
 MWI       Malawi                    Africa
 MOZ   Mozambique                    Africa
 ZAF South Africa                    Africa
 ZMB       Zambia                    Africa
 ZWE     Zimbabwe                    Africa
 AUS    Australia              Asia-Pacific
 BGD   Bangladesh              Asia-Pacific
 BTN       Bhutan              Asia-Pacific
 FJI         Fiji              Asia-Pacific
 LAO         Laos              Asia-Pacific
 NPL        Nepal              Asia-Pacific
 TWN       Taiwan              Asia-Pacific
 ARG    Argentina Latin America & Caribbean
 BOL      Bolivia Latin America & Caribbean
 CUB         Cuba Latin America & Caribbean
 GUY       Guyana Latin America & Caribbean


In [29]:
# Outcome: oil palm deforestation as a percentage of 2000 tree cover
df["palm_rate"] = df["defor_ha"] / df["extent_2000_ha"] * 100
df["y_palm_asinh"] = np.arcsinh(df["palm_rate"])
df["y_palm_log1p"] = np.log1p(df["palm_rate"])  # robustness transform

df[["palm_rate", "y_palm_asinh", "y_palm_log1p"]].describe().T

,count,mean,std,min,25%,50%,75%,max
palm_rate,1426.0,0.012815,0.049880,0.0,0.0,0.000037,0.003253,0.552801
y_palm_asinh,1426.0,0.012672,0.048759,0.0,0.0,0.000037,0.003253,0.527933
y_palm_log1p,1426.0,0.011727,0.042897,0.0,0.0,0.000037,0.003248,0.440060


In [30]:
# Summary by region and for the full sample
def summarise(group):
    return pd.Series({
        "countries": group["iso3"].nunique(),
        "country_years": len(group),
        "mean_rate_pct": group["palm_rate"].mean(),
        "median_rate_pct": group["palm_rate"].median(),
        "mean_asinh": group["y_palm_asinh"].mean(),
        "median_asinh": group["y_palm_asinh"].median(),
        "share_zeros": (group["palm_rate"] == 0).mean(),
        "share_zeros_reported_rows": (group.loc[group["in_deduce_palm"], "palm_rate"] == 0).mean(),
    })

by_region = pd.DataFrame({region: summarise(g) for region, g in df.groupby("region")}).T
by_region.loc["All countries"] = summarise(df)
by_region[["countries", "country_years"]] = by_region[["countries", "country_years"]].astype(int)
by_region.index.name = "region"
by_region.style.format({
    "mean_rate_pct": "{:.4f}", "median_rate_pct": "{:.5f}",
    "mean_asinh": "{:.4f}", "median_asinh": "{:.5f}",
    "share_zeros": "{:.1%}", "share_zeros_reported_rows": "{:.1%}",
})

,countries,country_years,mean_rate_pct,median_rate_pct,mean_asinh,median_asinh,share_zeros,share_zeros_reported_rows
region,,,,,,,,
Africa,24,552,0.0065,0.00001,0.0064,0.00001,43.1%,19.7%
Asia-Pacific,19,437,0.0264,0.00000,0.0260,0.00000,46.7%,15.6%
Latin America & Caribbean,19,437,0.0072,0.00034,0.0072,0.00034,33.0%,15.1%
All countries,62,1426,0.0128,0.00004,0.0127,0.00004,41.1%,17.0%


## 2. Baseline exposure

In [31]:
import os

import matplotlib.pyplot as plt
import wbgapi as wb

PROCESSED_OUT = "../data/processed/"
FIGURES = "../figures/"

In [32]:
# FAOSTAT oil palm area harvested, mean of the available years in 1996-2000
fao = pd.read_csv(BASE + "raw/crops/faostat_palm_area_1996_2000.csv", encoding="utf-8-sig")
assert set(fao["Element"]) == {"Area harvested"} and set(fao["Unit"]) == {"ha"}
assert fao["Year"].between(1996, 2000).all()

# FAOSTAT names that differ from the panel names
FAO_TO_PANEL = {
    "Mexico": "México",
    "Congo": "Republic of the Congo",
    "United Republic of Tanzania": "Tanzania",
    "Venezuela (Bolivarian Republic of)": "Venezuela",
}
fao["country"] = fao["Area"].replace(FAO_TO_PANEL)

palm_area = (fao.groupby("country")["Value"].mean()
                .rename("area_1996_2000_ha").reset_index())

sample = df[["iso3", "country", "region"]].drop_duplicates()
outside = sorted(set(palm_area["country"]) - set(sample["country"]))
print("FAOSTAT areas outside the 62-country sample:", outside)
assert set(outside) == {"Benin", "Burundi", "China", "China, mainland", "Gambia",
                        "Sao Tome and Principe", "Senegal", "Togo"}

HTTPError: HTTP Error 404: Not Found

In [ ]:
# World Bank land area in 2000 (km2 converted to ha)
land = wb.data.DataFrame("AG.LND.TOTL.K2", "all", time=2000, labels=False).iloc[:, 0]
land = (land.mul(100).rename("land_area_ha").rename_axis("iso3").reset_index())

exposure = (sample.merge(palm_area, on="country", how="left", validate="one_to_one")
                  .merge(land, on="iso3", how="left", validate="one_to_one"))

# Countries with no palm area are controls with exposure 0
exposure["area_1996_2000_ha"] = exposure["area_1996_2000_ha"].fillna(0.0)
exposure["exposure_palm"] = np.where(exposure["area_1996_2000_ha"].eq(0), 0.0,
                                     exposure["area_1996_2000_ha"] / exposure["land_area_ha"])

no_land = exposure.loc[exposure["land_area_ha"].isna(), "country"].tolist()
print("No World Bank land area:", no_land)
assert exposure.loc[exposure["area_1996_2000_ha"].gt(0), "land_area_ha"].notna().all()
assert exposure["exposure_palm"].notna().all() and len(exposure) == 62

exposure = exposure.sort_values("iso3").reset_index(drop=True)
EXPOSURE_COLS = ["iso3", "country", "region", "area_1996_2000_ha", "land_area_ha", "exposure_palm"]
if os.path.isdir(PROCESSED_OUT):
    exposure[EXPOSURE_COLS].to_csv(PROCESSED_OUT + "palm_exposure.csv", index=False)

n_exposed = int(exposure["exposure_palm"].gt(0).sum())
print("Countries with exposure > 0:", n_exposed, "of", len(exposure))
print(exposure.loc[exposure["exposure_palm"].gt(0)].groupby("region").size().to_string())

In [ ]:
# Top 15 countries by baseline exposure
top = exposure.nlargest(15, "exposure_palm").sort_values("exposure_palm")
values = top["exposure_palm"] * 100

fig, ax = plt.subplots(figsize=(9, 6.5))
ax.barh(top["country"], values, color="#2a78d6", height=0.7)
for y, value in enumerate(values):
    ax.text(value + values.max() * 0.01, y, f"{value:.2f}", va="center", fontsize=9, color="#333333")

ax.set_title(f"Baseline oil palm exposure: top 15 of the {n_exposed} exposed countries",
             loc="left", fontsize=13)
ax.set_xlabel("Oil palm area harvested, mean 1996-2000 (% of land area in 2000)")
ax.set_ylabel("Country")
ax.set_xlim(0, values.max() * 1.1)
ax.grid(axis="x", color="#e6e6e6", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)

fig.text(0.01, 0.005, "Source: FAOSTAT (QCL, oil palm fruit, area harvested); World Bank WDI (AG.LND.TOTL.K2).",
         fontsize=9, color="#666666")
plt.tight_layout(rect=(0, 0.03, 1, 1))
if os.path.isdir(FIGURES):
    fig.savefig(FIGURES + "palm_exposure_top15.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Consistency of the two palm sources and quality of the FAOSTAT figures
total_defor = df.groupby("iso3", as_index=False)["defor_ha"].sum().rename(columns={"defor_ha": "palm_defor_2001_2023_ha"})
check = exposure.merge(total_defor, on="iso3")

zero_with_defor = check[check["exposure_palm"].eq(0) & check["palm_defor_2001_2023_ha"].gt(0)]
print("Exposure 0 but positive DeDuCE oil palm deforestation:", len(zero_with_defor))
print(zero_with_defor[["country", "region", "palm_defor_2001_2023_ha"]]
      .sort_values("palm_defor_2001_2023_ha", ascending=False).to_string(index=False))
print("Their share of sample oil palm deforestation:",
      round(zero_with_defor["palm_defor_2001_2023_ha"].sum() / check["palm_defor_2001_2023_ha"].sum(), 4))

official = fao.assign(official=fao["Flag"].eq("A")).groupby("country")["official"].sum()
print(top.sort_values("exposure_palm", ascending=False).assign(
    exposure_pct=lambda d: (d["exposure_palm"] * 100).round(3),
    official_years_of_5=lambda d: d["country"].map(official).astype(int),
)[["country", "region", "area_1996_2000_ha", "exposure_pct", "official_years_of_5"]].to_string(index=False))